In [2]:
#Imports
import torch
import open_clip
from huggingface_hub import hf_hub_download
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight


### INVESTIGITON
Before doing the caching, we need to investigate the model's architecture and find what is happening to the two modality input. For fusion encoder, we need the sequences for both modalities because if we get the pooled vector result, we would implement the cross-attention to 2 different vectors, which makes no sense. We need to extract the sequences before the pooling phase so we can use the embeddings of 2 different modalities on cross-attention.

In [ ]:

device = "mps" if torch.mps.is_available() else "cpu"
print(f"[env] torch {torch.__version__} | open_clip {open_clip.__version__} | device {device}")


# Load RemoteCLIP

path = hf_hub_download("chendelong/RemoteCLIP", filename="RemoteCLIP-ViT-B-32.pt")
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained=path)
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model = model.to(device).eval()
print(" model loaded")


#  Map the module tree

print("\n" + "=" * 60)
print(" VISUAL tower attributes:")
print("=" * 60)
v = model.visual
for name, _ in v.named_children():
    print("  visual.", name, sep="")

print("\n key projection / pooling attributes present on model:")
for attr in ["text_projection", "logit_scale", "token_embedding", "ln_final", "positional_embedding"]:
    print(f"  model.{attr:20s} -> {'present' if hasattr(model, attr) else 'ABSENT'}")
for attr in ["proj", "ln_post", "class_embedding", "positional_embedding", "transformer", "attnpool"]:
    print(f"  model.visual.{attr:20s} -> {'present' if hasattr(v, attr) else 'ABSENT'}")


# Pooled (normal) path


img_path = "DI725_project_dataset/images/0073.png"  
try:
    pil = Image.open(img_path).convert("RGB")
    img_tensor = preprocess(pil).unsqueeze(0).to(device)
except FileNotFoundError:
    print(f"\n[stage2] WARNING: {img_path} not found, using random tensor for shape probe only")
    img_tensor = torch.randn(1, 3, 224, 224, device=device)

txt_tokens = tokenizer(["aerial view of open grassland and meadow"]).to(device)

with torch.no_grad():
    pooled_img = model.encode_image(img_tensor)
    pooled_txt = model.encode_text(txt_tokens)
print("\n[stage2] pooled_img", tuple(pooled_img.shape), "| pooled_txt", tuple(pooled_txt.shape))


#  Reach the PRE-POOL sequences via forward hooks to determine if we can use the embeddings for training frozen encoders
captured = {}

def grab(name):
    def hook(module, inp, out):
        captured[name] = out
    return hook

# VISION: the transformer inside the visual tower outputs the patch
# sequence BEFORE final ln_post/proj pooling
handles = []
if hasattr(v, "transformer"):
    handles.append(v.transformer.register_forward_hook(grab("vision_seq")))
# TEXT: the top-level transformer processes the token sequence before
# ln_final + text_projection pool it to a vector
if hasattr(model, "transformer"):
    handles.append(model.transformer.register_forward_hook(grab("text_seq")))

with torch.no_grad():
    _ = model.encode_image(img_tensor)
    _ = model.encode_text(txt_tokens)

for h in handles:
    h.remove()

print("\n captured keys:", list(captured.keys()))
for k, val in captured.items():
    t = val[0] if isinstance(val, tuple) else val
    # open_clip transformers are often [seq, batch, width] (seq-first) 
    print(f"  {k}: shape {tuple(t.shape)}  (watch for seq-first vs batch-first)")


#Dimension reckoning

print("\nwidth comparison:")
for k, val in captured.items():
    t = val[0] if isinstance(val, tuple) else val
    print(f"  {k} last-dim (sequence width) = {t.shape[-1]}")
print(f"  pooled/projected width = {pooled_img.shape[-1]}  (this is the 512-d template space)")
print("  -> if sequence width != 512, Option A's fusion layer needs an internal projection")


[env] torch 2.12.0 | open_clip 3.3.0 | device mps
 model loaded

 VISUAL tower attributes:
  visual.conv1
  visual.patch_dropout
  visual.ln_pre
  visual.transformer
  visual.ln_post

 key projection / pooling attributes present on model:
  model.text_projection      -> present
  model.logit_scale          -> present
  model.token_embedding      -> present
  model.ln_final             -> present
  model.positional_embedding -> present
  model.visual.proj                 -> present
  model.visual.ln_post              -> present
  model.visual.class_embedding      -> present
  model.visual.positional_embedding -> present
  model.visual.transformer          -> present
  model.visual.attnpool             -> ABSENT

[stage2] WARNING: DI725_project_dataset/images/0073.png not found, using random tensor for shape probe only

[stage2] pooled_img (1, 512) | pooled_txt (1, 512)

 captured keys: ['vision_seq', 'text_seq']
  vision_seq: shape (1, 50, 768)  (watch for seq-first vs batch-first)
  te

### Caching
Since the RemoteCLIP encoders are frozen (not trained), their output embeddings never change. We therefore run the vision and text encoders once over the whole dataset and cache the resulting embeddings to disk, instead of recomputing them every epoch. Only the fusion module and the classification head are trained, so caching the frozen encoder outputs makes training fast, each epoch reads precomputed vectors rather than running the heavy transformers again.

In [4]:



device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 64
OUT_PATH = "output/aras400k_cache.pt"
INPUT_CSV = "/Users/egeardaozturk/Downloads/DI725_project_dataset/captions.csv"
INPUT_IMG = "/Users/egeardaozturk/Downloads/DI725_project_dataset/images/"
INPUT_MASK = "/Users/egeardaozturk/Downloads/DI725_project_dataset/masks/"
class_columns = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
class_names = ['tree', 'shrub', 'grass', 'crop', 'built-up', 'barren', 'water']


# Load RemoteCLIP (frozen)

path = hf_hub_download("chendelong/RemoteCLIP", filename="RemoteCLIP-ViT-B-32.pt")
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained=path)
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model = model.to(device).eval()


df = pd.read_csv(INPUT_CSV)
image_paths = df['filename'].apply(lambda x: INPUT_IMG + x).tolist()  
correct_caps  = df['hybrid_gemma3-4b'].tolist()
df['dominant_class_idx'] = df[class_columns].values.argmax(axis=1)
df['dominant_class'] = df['dominant_class_idx'].apply(lambda x: class_names[x])
labels_np = df['dominant_class_idx'].values
CLASS_NAMES = ["Tree","Shrub","Grass","Crop","Built-up","Barren","Water"]
COMPOSITION_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]
CAPTION_COL = "hybrid_gemma3-4b"
CLASS_NAMES = COMPOSITION_COLS  # index i <-> CLASS_NAMES[i]


df["dominant_class"] = df[COMPOSITION_COLS].values.argmax(axis=1)  # 0..6 int
 
# class distribution 
print("class distribution:")
print(df["dominant_class"].value_counts().sort_index()
        .rename(lambda i: CLASS_NAMES[i]))

np.random.seed(42)
 
# Pre-bucket row indices by dominant class so sampling is fast.
idx_by_class = {c: df.index[df["dominant_class"] == c].to_numpy()
                for c in range(len(CLASS_NAMES))}
all_classes = np.array([c for c in idx_by_class if len(idx_by_class[c]) > 0])
 
misleading = []
for _, row in df.iterrows():
    cur = row["dominant_class"]
    # pick a class different from this row's, that actually has members
    other_classes = all_classes[all_classes != cur]
    chosen_class = np.random.choice(other_classes)
    donor_idx = np.random.choice(idx_by_class[chosen_class])
    misleading.append(df.at[donor_idx, CAPTION_COL])
 
df["misleading_caption"] = misleading
df["correct_caption"] = df[CAPTION_COL]
 

violations = (df["misleading_caption"].values == df["correct_caption"].values).sum()
print(f"\nrows where misleading == correct (should be ~0): {violations}")
 



#  capture pre-pool sequences during the same forward pass

_cap = {}
def _grab(name):
    def hook(m, i, o):
        _cap[name] = (o[0] if isinstance(o, tuple) else o).detach()
    return hook
h1 = model.visual.transformer.register_forward_hook(_grab("vseq"))
h2 = model.transformer.register_forward_hook(_grab("tseq"))


# Allocate output tensors (fp16)
N = len(image_paths)  

vision_seq = torch.empty(N, 50, 768, dtype=torch.float16)
pooled_img = torch.empty(N, 512, dtype=torch.float16)
text_seq_c = torch.empty(N, 77, 512, dtype=torch.float16)
text_seq_m = torch.empty(N, 77, 512, dtype=torch.float16)
pooled_txt_c = torch.empty(N, 512, dtype=torch.float16)
pooled_txt_m = torch.empty(N, 512, dtype=torch.float16)

def encode_image_batch(pil_batch):
    imgs = torch.stack([preprocess(im) for im in pil_batch]).to(device)
    with torch.no_grad():
        pooled = model.encode_image(imgs)      # [B,512], triggers vseq hook
    return _cap["vseq"].cpu(), pooled.cpu()    # seq [B,50,768], pooled [B,512]

def encode_text_batch(caps):
    toks = tokenizer(caps).to(device)
    with torch.no_grad():
        pooled = model.encode_text(toks)       # [B,512], triggers tseq hook
    return _cap["tseq"].cpu(), pooled.cpu()    # seq [B,77,512], pooled [B,512]


# Single pass — fixed order, NO shuffling

for s in range(0, N, BATCH):
    e = min(s + BATCH, N)
    pil = [Image.open(p).convert("RGB") for p in image_paths[s:e]]  
    vseq, pimg = encode_image_batch(pil)
    vision_seq[s:e] = vseq.half();  pooled_img[s:e] = pimg.half()

    tcs, tcp = encode_text_batch(correct_caps[s:e])     
    text_seq_c[s:e] = tcs.half();  pooled_txt_c[s:e] = tcp.half()

    tms, tmp = encode_text_batch(misleading[s:e]) 
    text_seq_m[s:e] = tms.half();  pooled_txt_m[s:e] = tmp.half()

    if s % (BATCH * 20) == 0:
        print(f"  {e}/{N}")

h1.remove(); h2.remove()


# Stratified split once, store indices, never reorder the cache
labels = torch.tensor(labels_np, dtype=torch.long)  # noqa: F821
idx = np.arange(N)
train_idx, test_idx = train_test_split(
    idx, test_size=0.30, stratify=labels_np, random_state=42)  



Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'vision_seq': vision_seq,
    'text_seq_correct': text_seq_c,
    'text_seq_misleading': text_seq_m,
    'pooled_img': pooled_img,
    'pooled_txt_correct': pooled_txt_c,
    'pooled_txt_misleading': pooled_txt_m,
    'labels': labels,
    'train_idx': torch.tensor(train_idx),
    'test_idx': torch.tensor(test_idx),
    'class_names': CLASS_NAMES,
    'note': 'raw/unnormalized; L2-norm pooled vecs at use, identically per arm',
}, OUT_PATH)
print(f"saved -> {OUT_PATH}")



class distribution:
dominant_class
Tree        3037
Shrub         27
Grass       4703
Crop        1803
Built-up      59
Barren       193
Water        178
Name: count, dtype: int64

rows where misleading == correct (should be ~0): 0
  64/10000
  1344/10000
  2624/10000
  3904/10000
  5184/10000
  6464/10000
  7744/10000
  9024/10000
saved -> output/aras400k_cache.pt


In [ ]:



#Irrelevant cache, it was not implemented before

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
N = 10000  # match your dataset size


# Irrelevant text = coherent sentences with NO land-cover content.
# Deng et al. use WikiText passages
IRRELEVANT_POOL = [
    "The quarterly financial report showed steady growth in consumer spending.",
    "She practiced the violin for three hours before the evening concert.",
    "The recipe calls for two cups of flour and a pinch of salt.",
    "Local transit authorities announced new bus schedules for the winter.",
    "The novel explores themes of memory and identity across generations.",
    "Researchers presented their findings at the annual linguistics conference.",
    "The museum's new exhibit features artifacts from the bronze age.",
    "He repaired the bicycle chain and adjusted the brake cables.",
    "The committee voted to extend the library's weekend opening hours.",
    "A gentle melody drifted from the cafe on the corner of the street.",
]

np.random.seed(42)
irrelevant_caps = [IRRELEVANT_POOL[np.random.randint(len(IRRELEVANT_POOL))]
                   for _ in range(N)]


# Encode with the SAME frozen RemoteCLIP + same hook as the cache,
# so text_seq_irrelevant has identical shape/space to the others.

path = hf_hub_download("chendelong/RemoteCLIP", filename="RemoteCLIP-ViT-B-32.pt")
model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained=path)
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model = model.to(device).eval()

_cap = {}
def _grab(name):
    def hook(m, i, o):
        _cap[name] = (o[0] if isinstance(o, tuple) else o).detach()
    return hook
h = model.transformer.register_forward_hook(_grab("tseq"))

BATCH = 64
text_seq_irr  = torch.empty(N, 77, 512, dtype=torch.float16)
pooled_txt_irr = torch.empty(N, 512, dtype=torch.float16)
for s in range(0, N, BATCH):
    e = min(s + BATCH, N)
    toks = tokenizer(irrelevant_caps[s:e]).to(device)
    with torch.no_grad():
        pooled = model.encode_text(toks)
    text_seq_irr[s:e]  = _cap["tseq"].cpu().half()
    pooled_txt_irr[s:e] = pooled.cpu().half()
h.remove()

# Append to existing cache (or save separately and load alongside)
cache = torch.load("output/aras400k_cache.pt")
cache["text_seq_irrelevant"]    = text_seq_irr
cache["pooled_txt_irrelevant"]  = pooled_txt_irr
torch.save(cache, "output/aras400k_cache.pt")
print("added irrelevant condition to cache")



added irrelevant condition to cache


### CHECK PHASE-2 ACCURACY WITH CACHED ENCODER
Checking the zero-shot classification accuracy between the RemoteCLIP used in Phase-2 and the cached frozen RemoteCLIP encoders, so we can see if we have implemented caching accurate.

In [6]:


device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")


# Load cache

cache = torch.load("output/aras400k_cache.pt")
pooled_img = cache["pooled_img"].float()      # [N,512] (cast fp16 -> fp32)
labels     = cache["labels"]                  # [N]
class_names = cache["class_names"]            # 7 names



class_templates = [
    "aerial view of dense tree cover and forest canopy",
    "aerial view of low shrubs and scrubland vegetation",
    "aerial view of open grassland and meadow",
    "aerial view of agricultural cropland and farmland",
    "aerial view of urban buildings and roads",
    "aerial view of barren desert and bare soil",
    "aerial view of river lake or water surface",
]


# Encode templates with the SAME frozen RemoteCLIP

path = hf_hub_download("chendelong/RemoteCLIP", filename="RemoteCLIP-ViT-B-32.pt")
model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained=path)
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model = model.to(device).eval()

with torch.no_grad():
    tmpl_tokens = tokenizer(class_templates).to(device)
    tmpl_emb = model.encode_text(tmpl_tokens).float().cpu()   # [7,512]


# THE EVAL PATH 
#   normalize -> cosine similarity -> argmax -> score
# L2-normalize BOTH sides so dot product == cosine similarity.
img_n  = torch.nn.functional.normalize(pooled_img, dim=-1)   # [N,512]
tmpl_n = torch.nn.functional.normalize(tmpl_emb,  dim=-1)    # [7,512]

sims = img_n @ tmpl_n.T          # [N,7] cosine similarity to each template
preds = sims.argmax(dim=1)       # [N] predicted class

y_true = labels.numpy()
y_pred = preds.numpy()


# Scores — compare ACC to Phase-2 RemoteCLIP ~15.95%

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"zero-shot accuracy : {acc*100:.2f}%   (Phase-2 RemoteCLIP was ~15.95%)")
print(f"zero-shot macro F1 : {macro_f1:.4f}")
print("\nper-class report:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

zero-shot accuracy : 15.95%   (Phase-2 RemoteCLIP was ~15.95%)
zero-shot macro F1 : 0.2340

per-class report:
              precision    recall  f1-score   support

        Tree       0.94      0.15      0.26      3037
       Shrub       0.01      0.52      0.01        27
       Grass       0.14      0.04      0.06      4703
        Crop       0.64      0.31      0.42      1803
    Built-up       0.54      0.37      0.44        59
      Barren       0.05      0.93      0.09       193
       Water       0.22      0.96      0.36       178

    accuracy                           0.16     10000
   macro avg       0.36      0.47      0.23     10000
weighted avg       0.47      0.16      0.19     10000



In [7]:




# Two arms, ONE shared head. The only difference between them is
# how the image representation is produced before the head.
# That single-variable difference is the ablation.



class FusionEncoder(nn.Module):
    """
    Image patches (Q) cross-attend over text tokens (K,V).
    The misleading caption therefore has a direct path into the
    image representation -- the blind-faith pathway you're testing.

    Shapes (from your verified cache):
      vision_seq : [B, 50, 768]
      text_seq   : [B, 77, 512]
    """
    def __init__(self, vision_width=768, embed_dim=512, num_heads=8, num_classes=7):
        super().__init__()
        # 768 -> 512 so image patches live in the same space as text + templates
        self.vision_proj = nn.Linear(vision_width, embed_dim)
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        # SHARED head, must be identical to the dual arm's head
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, vision_seq, text_seq):
        # vision_seq: [B,50,768]   text_seq: [B,77,512]
        #

        projected_vision = self.vision_proj(vision_seq)

        cross_attn_out, _ = self.cross_attn(query=projected_vision, key=text_seq, value=text_seq)

        result = self.norm(cross_attn_out + projected_vision)
        pooled = result.mean(dim=1)
        # Return the logits. Keep every op in 512-d.
        return self.head(pooled)

class DualEncoder(nn.Module):
    """
    Control arm. Pooled image vector -> the SAME head.
    Text never enters the representation. This is late fusion:
    text has no path to corrupt the image vector.
    """
    def __init__(self, embed_dim=512, num_classes=7):
        super().__init__()
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, pooled_img):
        # pooled_img: [B,512] straight from the cache
        return self.head(pooled_img)

m = FusionEncoder()
out = m(torch.randn(4, 50, 768), torch.randn(4, 77, 512))
print(out.shape)  




torch.Size([4, 7])


### TRAINING
Both arms are trained on matching captions only, then evaluated under matching, corrupted, and irrelevant captions. Details are in the code comments.

In [8]:




EPOCHS = 15
LR = 1e-3
BATCH = 256
WEIGHT_DECAY = 1e-4
SEED = 42
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")


torch.manual_seed(SEED); np.random.seed(SEED)


# Load cache 
c = torch.load("output/aras400k_cache.pt")
labels      = c["labels"]
train_idx   = c["train_idx"]
test_idx    = c["test_idx"]
class_names = c["class_names"]
NUM_CLASSES = len(class_names)

vision_seq  = c["vision_seq"].float()
pooled_img  = c["pooled_img"].float()

# three text conditions
text_cond = {
    "match":       c["text_seq_correct"].float(),
    "corruption":  c["text_seq_misleading"].float(),
    "irrelevance": c["text_seq_irrelevant"].float(),
}

# Class weights from TRAIN labels only
train_labels = labels[train_idx].numpy()
present = np.unique(train_labels)
cw = compute_class_weight("balanced", classes=present, y=train_labels)
weight_vec = torch.ones(NUM_CLASSES)
for cls, w in zip(present, cw):
    weight_vec[cls] = w
criterion = nn.CrossEntropyLoss(weight=weight_vec.to(device))


def evaluate(model, arm, text_seq, split_idx):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for i in range(0, len(split_idx), BATCH):
            b = split_idx[i:i+BATCH]
            if arm == "fusion":
                logits = model(vision_seq[b].to(device), text_seq[b].to(device))
            else:
                logits = model(pooled_img[b].to(device))
            ps.append(logits.argmax(1).cpu()); ys.append(labels[b])
    y = torch.cat(ys).numpy(); p = torch.cat(ps).numpy()
    return f1_score(y, p, average="macro", zero_division=0), accuracy_score(y, p), y, p


def train_arm(arm):
    model = (FusionEncoder(num_classes=NUM_CLASSES) if arm == "fusion"
             else DualEncoder(num_classes=NUM_CLASSES)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    idx_t = train_idx[torch.randperm(len(train_idx))]
    for ep in range(EPOCHS):
        model.train()
        idx_t = idx_t[torch.randperm(len(idx_t))]
        running = 0.0
        for i in range(0, len(idx_t), BATCH):
            b = idx_t[i:i+BATCH]
            if arm == "fusion":
                logits = model(vision_seq[b].to(device), text_cond["match"][b].to(device))
            else:
                logits = model(pooled_img[b].to(device))
            loss = criterion(logits, labels[b].to(device))
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item()
        f1_val, _, _, _ = evaluate(model, arm, text_cond["match"], test_idx)
        print(f"[{arm}] epoch {ep+1:2d}  loss {running/(len(idx_t)//BATCH+1):.3f}  val_macroF1(match) {f1_val:.4f}")

    return model



# Run both arms across all THREE text conditions.

print("=" * 64)
results = {}
for arm in ["dual", "fusion"]:
    model = train_arm(arm)
    results[arm] = {}
    print(f"\n=== {arm.upper()} ===")
    for cond, text_seq in text_cond.items():
        f1, acc, y, p = evaluate(model, arm, text_seq, split_idx=test_idx)
        results[arm][cond] = (f1, acc)
        print(f"  {cond:12s}: macroF1 {f1:.4f}  acc {acc*100:.2f}%")
    _, _, yc, pc = evaluate(model, arm, text_cond["corruption"], test_idx)
    print("  per-class (corruption):")
    print(classification_report(yc, pc, target_names=class_names, zero_division=0))


# Comparison table -- the three-leg result

print("=" * 64)
print(f"{'':8s}{'match':>12s}{'corruption':>14s}{'irrelevance':>14s}")
for arm in ["dual", "fusion"]:
    m = results[arm]["match"][0]
    cor = results[arm]["corruption"][0]
    irr = results[arm]["irrelevance"][0]
    print(f"{arm:8s}{m:>12.4f}{cor:>14.4f}{irr:>14.4f}")

print("\nReading the fusion row:")
print("  match high, corruption collapses, irrelevance near dual baseline")
print("    -> blind faith: only plausible WRONG text misleads; image used otherwise")
print("  match high, BOTH corruption and irrelevance collapse")
print("    -> image-blind shortcut: fusion ignores image regardless of text validity")
print("  (dual row identical across columns: it never reads text)")

[dual] epoch  1  loss 1.574  val_macroF1(match) 0.5285
[dual] epoch  2  loss 1.032  val_macroF1(match) 0.5343
[dual] epoch  3  loss 0.799  val_macroF1(match) 0.5498
[dual] epoch  4  loss 0.691  val_macroF1(match) 0.5567
[dual] epoch  5  loss 0.606  val_macroF1(match) 0.5691
[dual] epoch  6  loss 0.563  val_macroF1(match) 0.5674
[dual] epoch  7  loss 0.520  val_macroF1(match) 0.5764
[dual] epoch  8  loss 0.496  val_macroF1(match) 0.5841
[dual] epoch  9  loss 0.462  val_macroF1(match) 0.5889
[dual] epoch 10  loss 0.447  val_macroF1(match) 0.5894
[dual] epoch 11  loss 0.424  val_macroF1(match) 0.5935
[dual] epoch 12  loss 0.406  val_macroF1(match) 0.5964
[dual] epoch 13  loss 0.395  val_macroF1(match) 0.6010
[dual] epoch 14  loss 0.382  val_macroF1(match) 0.6014
[dual] epoch 15  loss 0.381  val_macroF1(match) 0.5997

=== DUAL ===
  match       : macroF1 0.5997  acc 77.40%
  corruption  : macroF1 0.5997  acc 77.40%
  irrelevance : macroF1 0.5997  acc 77.40%
  per-class (corruption):
      